In [5]:
import gc
import time
from collections import Counter

import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

In [6]:
TRAIN_IN = "/kaggle/input/datasets/zerotoinfinitya/train-preprocessed/train_preprocessed.parquet"
VAL_IN = "/kaggle/input/datasets/zerotoinfinitya/validation-preprocessed/validation_preprocessed.parquet"
TEST_IN = "/kaggle/input/datasets/zerotoinfinitya/test-preprocessed/test_preprocessed.parquet"

TRAIN_OUT = "/kaggle/working/train_bytecode_features.npz"
VAL_OUT = "/kaggle/working/validation_bytecode_features.npz"
TEST_OUT = "/kaggle/working/test_bytecode_features.npz"
VECTORIZER_OUT = "/kaggle/working/vectorizer_bytecode_12k.joblib"
SCALER_OUT = "/kaggle/working/scaler_bytecode.joblib"

In [7]:
def read_opcodes(path):
    df = pd.read_parquet(path, columns=["opcodes"])
    return ["" if x is None else x for x in df["opcodes"]]


In [8]:
RISK_OPS = ["CALL", "DELEGATECALL", "CALLCODE", "STATICCALL", "CREATE",
            "CREATE2", "SELFDESTRUCT", "ORIGIN", "CALLER", "CALLVALUE",
            "BALANCE", "TIMESTAMP", "BLOCKHASH", "DIFFICULTY", "NUMBER",
            "COINBASE", "GASLIMIT", "SLOAD", "SSTORE", "JUMP", "JUMPI",
            "JUMPDEST", "REVERT", "INVALID", "ADD", "SUB", "MUL", "DIV",
            "MOD", "EXP", "SDIV", "SMOD", "LT", "GT", "EQ", "ISZERO",
            "AND", "OR", "XOR", "NOT", "KECCAK256", "LOG0", "LOG1",
            "LOG2", "LOG3", "EXTCODESIZE", "RETURNDATASIZE", "GAS",
            "PREVRANDAO", "SELFBALANCE", "STOP", "RETURN"]
DERIVED = ["n_ops_k", "diversity", "unk_rate", "empty",
           "call_then_sstore", "sstore_then_call", "call_in_block",
           "n_blocks_k", "mean_block_len", "jumpi_rate"]

In [9]:
BCH_NAMES = [o.lower() for o in RISK_OPS] + DERIVED
BASE = len(RISK_OPS)
assert len(BCH_NAMES) == 62

In [13]:
def op_features(ops_txt):
    F = np.zeros((len(ops_txt), 62), dtype=np.float32)
    start = time.time()
    for i in range(len(ops_txt)):
        toks = (ops_txt[i] or "").split()
        if not toks:
            F[i, BASE + 3] = 1.0              # `empty`: no bytecode
            continue
        tot = len(toks)
        per_k = tot / 1000.0
        c = Counter(toks)

        # a RATE, so a big contract does not look risky just for size
        for j in range(BASE):
            F[i, j] = c.get(RISK_OPS[j], 0) / per_k
        F[i, BASE + 0] = tot / 1000.0
        F[i, BASE + 1] = len(c) / 100.0
        F[i, BASE + 2] = c.get("UNK", 0) / per_k

        # cut the stream into blocks at every JUMPDEST, then ask:
        # inside one block, does a CALL come before an SSTORE?
        # That is the reentrancy shape, seen in compiled code.
        ct_s, st_c, cib, blocks, lens = 0, 0, 0, 0, 0
        cur = []
        CALLS = ("CALL", "DELEGATECALL", "CALLCODE", "STATICCALL")
        for t in toks + ["JUMPDEST"]:
            if t == "JUMPDEST":
                if cur:
                    blocks += 1
                    lens += len(cur)
                    ci = [k for k, x in enumerate(cur) if x in CALLS]
                    si = [k for k, x in enumerate(cur) if x == "SSTORE"]
                    if ci:
                        cib += 1
                        if si and max(si) > min(ci):
                            ct_s += 1
                        if si and min(si) < max(ci):
                            st_c += 1
                    cur = []
            else:
                cur.append(t)
        F[i, BASE + 4] = ct_s
        F[i, BASE + 5] = st_c
        F[i, BASE + 6] = cib
        F[i, BASE + 7] = blocks / 1000.0
        F[i, BASE + 8] = lens / max(1, blocks)
        F[i, BASE + 9] = c.get("JUMPI", 0) / per_k

        if i and i % 20000 == 0:
            print("   %d / %d   %d s" % (i, len(ops_txt), time.time() - start))
    return F

In [14]:
vectorizer = TfidfVectorizer(
    max_features=12000,       
    ngram_range=(1, 3),       
    min_df=5,                 
    token_pattern=r"\S+",     
    sublinear_tf=True,
)

In [15]:
train_ops = read_opcodes(TRAIN_IN)
vectorizer.fit(train_ops)
vocab = np.asarray([str(t) for t in vectorizer.get_feature_names_out()],
                   dtype=np.str_)
print("vocabulary:", len(vocab), "n-grams")

vocabulary: 12000 n-grams


In [16]:
train_bcops = vectorizer.transform(train_ops).astype(np.float32).tocsr()
train_raw = op_features(train_ops)
example_ops = train_ops[0]           
gc.collect()

   20000 / 79641   29 s
   40000 / 79641   59 s
   60000 / 79641   89 s


1156

In [17]:
val_ops = read_opcodes(VAL_IN)
val_bcops = vectorizer.transform(val_ops).astype(np.float32).tocsr()
val_raw = op_features(val_ops)
del val_ops
gc.collect()

0

In [18]:
test_ops = read_opcodes(TEST_IN)
test_bcops = vectorizer.transform(test_ops).astype(np.float32).tocsr()
test_raw = op_features(test_ops)
del test_ops
gc.collect()

0

In [19]:
scaler = StandardScaler()
scaler.fit(train_raw)

StandardScaler()

In [20]:


train_hand = scaler.transform(train_raw).astype(np.float32)
val_hand = scaler.transform(val_raw).astype(np.float32)
test_hand = scaler.transform(test_raw).astype(np.float32)


In [21]:
def save(path, bcops, hand):
    np.savez_compressed(path,
                        bcops_data=bcops.data,
                        bcops_indices=bcops.indices,
                        bcops_indptr=bcops.indptr,
                        bcops_shape=np.asarray(bcops.shape),
                        bcops_vocabulary=vocab,
                        bchand=hand,
                        bchand_names=np.asarray(BCH_NAMES, dtype=np.str_))


save(TRAIN_OUT, train_bcops, train_hand)
save(VAL_OUT, val_bcops, val_hand)
save(TEST_OUT, test_bcops, test_hand)
joblib.dump(vectorizer, VECTORIZER_OUT, compress=3)
joblib.dump({"scaler": scaler, "names": BCH_NAMES}, SCALER_OUT)

print("  ", TRAIN_OUT)
print("  ", VAL_OUT)
print("  ", TEST_OUT)
print("  ", VECTORIZER_OUT)
print("  ", SCALER_OUT)


Saved:
   /kaggle/working/train_bytecode_features.npz
   /kaggle/working/validation_bytecode_features.npz
   /kaggle/working/test_bytecode_features.npz
   /kaggle/working/vectorizer_bytecode_12k.joblib
   /kaggle/working/scaler_bytecode.joblib
